# Calling the Vision REST API

In this lab, you will send image analysis requests directly to the Azure AI Vision REST API using the `requests` library, parse the structured response for each visual feature, and handle common error scenarios.

## Learning Objectives

- Construct authenticated REST requests to the Azure AI Vision Image Analysis endpoint
- Request multiple visual features (tags, captions, objects, read) in a single API call
- Parse the JSON response and extract results for each visual feature
- Send both URL-based and binary image inputs

## Prerequisites

- An active Azure subscription
- A Foundry resource deployed in a region that supports Image Analysis 4.0 (for example, **East US**; caption, dense captions, and people detection are region-gated). This single resource provides the Vision endpoint — **no separate Azure AI Vision resource is required.
- Keyless authentication with Microsoft Entra ID (RBAC).** Newer Foundry resources ship with API keys disabled, so this lab authenticates with your Azure identity instead of a key. Two setup steps:
  1. Grant your identity the **Cognitive Services User** role on the resource. Being subscription *Owner* is **not** enough — that covers management (control plane), not the data-plane calls the API makes.
     ```bash
     # your object id:   az ad signed-in-user show --query id -o tsv
     # the resource id:  az cognitiveservices account show -n <resource> -g <rg> --query id -o tsv
     az role assignment create \
       --assignee-object-id <your-object-id> --assignee-principal-type User \
       --role "Cognitive Services User" \
       --scope <resource-id>
     ```
  2. Sign in so `DefaultAzureCredential` can obtain a token: `az login`
- Your resource endpoint (`https://<resource>.cognitiveservices.azure.com/`) in `.env` (copy `.env.example`); leave `AZURE_AI_VISION_KEY` blank to use Entra ID.

## 1 — Install Dependencies

In [ ]:
%pip install azure-identity python-dotenv requests --quiet


## 2 — Configure the Endpoint and Credential

Replace the placeholder values below with your Azure AI Vision resource endpoint. If no key is provided, the notebook uses `DefaultAzureCredential` with Microsoft Entra authentication.

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

load_dotenv()

VISION_ENDPOINT = os.environ["AZURE_AI_VISION_ENDPOINT"]
VISION_KEY = os.environ.get("AZURE_AI_VISION_KEY", "").strip()
if VISION_KEY.startswith("your-") or VISION_KEY.startswith("<"):
    VISION_KEY = ""

API_VERSION = "2024-02-01"
BASE_URL = f"{VISION_ENDPOINT}/computervision/imageanalysis:analyze"
TOKEN_CREDENTIAL = DefaultAzureCredential() if not VISION_KEY else None


def vision_headers() -> dict:
    headers = {"Content-Type": "application/json"}
    if VISION_KEY:
        headers["Ocp-Apim-Subscription-Key"] = VISION_KEY
    else:
        token = TOKEN_CREDENTIAL.get_token("https://cognitiveservices.azure.com/.default")
        headers["Authorization"] = f"Bearer {token.token}"
    return headers

print(f"Endpoint: {VISION_ENDPOINT}")
print(f"API version: {API_VERSION}")
print(f"Authentication: {'API key' if VISION_KEY else 'DefaultAzureCredential'}")


## 3 — Analyze an Image by URL

The simplest way to call the API is to pass an image URL in the request body. The service fetches the image from the URL and returns the requested visual features.

The `features` query parameter accepts a comma-separated list. Here we request tags, a caption, object detection, and OCR in a single call.

In [ ]:
import requests
import json

headers = vision_headers()

params = {
    "features": "tags,caption,objects,read",
    "api-version": API_VERSION,
}

# A sample image of a busy street scene
image_url = "https://learn.microsoft.com/en-us/azure/ai-services/computer-vision/media/quickstarts/presentation.png"

body = {"url": image_url}

response = requests.post(BASE_URL, headers=headers, params=params, json=body)

print(f"Status code: {response.status_code}")
print(json.dumps(response.json(), indent=2))


## 4 — Parse Tags and Captions

The response contains a `tagsResult` array and a `captionResult` object. Each tag includes a `name` and `confidence` score between 0.0 and 1.0.

In [ ]:
result = response.json()

# Parse caption
caption = result.get("captionResult", {})
print(f"Caption: {caption.get('text', 'N/A')}")
print(f"Caption confidence: {caption.get('confidence', 0):.4f}")
print()

# Parse tags — filter to high-confidence results
CONFIDENCE_THRESHOLD = 0.7
tags = result.get("tagsResult", {}).get("values", [])

print(f"Tags (confidence >= {CONFIDENCE_THRESHOLD}):")
for tag in tags:
    if tag["confidence"] >= CONFIDENCE_THRESHOLD:
        print(f"  {tag['name']:30s} confidence: {tag['confidence']:.4f}")

print(f"\nTotal tags returned: {len(tags)}")
print(f"Tags above threshold: {sum(1 for t in tags if t['confidence'] >= CONFIDENCE_THRESHOLD)}")


## 5 — Parse Object Detection Results

Each detected object includes a label, confidence score, and a bounding box with pixel coordinates (top, left, width, height).

In [ ]:
objects = result.get("objectsResult", {}).get("values", [])

print(f"Objects detected: {len(objects)}\n")
for obj in objects:
    tags_list = obj.get("tags", [])
    if tags_list:
        label = tags_list[0].get("name", "unknown")
        confidence = tags_list[0].get("confidence", 0)
    else:
        label = "unknown"
        confidence = 0

    bbox = obj.get("boundingBox", {})
    print(f"  {label:20s} confidence: {confidence:.4f}")
    print(f"    Bounding box: x={bbox.get('x', 0)}, y={bbox.get('y', 0)}, "
          f"w={bbox.get('w', 0)}, h={bbox.get('h', 0)}")


## 6 — Parse OCR Results

The Read feature returns a hierarchical structure: blocks, lines, and words. Each word includes its text content and a bounding polygon (four points, not a rectangle — this handles rotated text).

In [ ]:
read_result = result.get("readResult", {})
blocks = read_result.get("blocks", [])

print(f"Blocks detected: {len(blocks)}\n")

for block_idx, block in enumerate(blocks):
    lines = block.get("lines", [])
    print(f"Block {block_idx + 1} ({len(lines)} lines):")
    for line in lines:
        text = line.get("text", "")
        words = line.get("words", [])
        avg_confidence = (
            sum(w.get("confidence", 0) for w in words) / len(words)
            if words
            else 0
        )
        print(f"  \"{text}\"  (avg word confidence: {avg_confidence:.4f})")
    print()


## 7 — Analyze a Binary Image

Instead of passing a URL, you can send the raw image bytes directly. This is useful when the image is generated in memory, uploaded by a user, or stored locally.

The key difference: set `Content-Type` to `application/octet-stream` and send the bytes as the request body (not JSON). Authentication still uses the same helper, so this works with either an API key or Microsoft Entra bearer token.

In [ ]:
# Download a sample image to simulate having local bytes
sample_url = "https://learn.microsoft.com/en-us/azure/ai-services/computer-vision/media/quickstarts/presentation.png"
image_data = requests.get(sample_url).content
print(f"Downloaded {len(image_data):,} bytes\n")

# Send as binary
binary_headers = vision_headers()
binary_headers["Content-Type"] = "application/octet-stream"

binary_params = {
    "features": "caption,tags",
    "api-version": API_VERSION,
}

binary_response = requests.post(
    BASE_URL, headers=binary_headers, params=binary_params, data=image_data
)

binary_result = binary_response.json()
print(f"Status code: {binary_response.status_code}")
print(f"Caption: {binary_result.get('captionResult', {}).get('text', 'N/A')}")
print(f"Tags: {[t['name'] for t in binary_result.get('tagsResult', {}).get('values', [])[:5]]}")


## 8 — Request Dense Captions and People Detection

Dense captions produce descriptions for multiple regions in the image — each with its own bounding box. People detection returns person-specific bounding boxes, which is handled separately from general object detection.

In [ ]:
advanced_url = "https://learn.microsoft.com/en-us/azure/ai-services/computer-vision/media/quickstarts/presentation.png"

result = analyze_image_with_retry(
    advanced_url, features="denseCaptions,people"
)

# Dense captions
dense = result.get("denseCaptionsResult", {}).get("values", [])
print(f"Dense captions ({len(dense)} regions):\n")
for dc in dense:
    bbox = dc.get("boundingBox", {})
    print(f"  \"{dc.get('text', '')}\"")
    print(f"    Confidence: {dc.get('confidence', 0):.4f}")
    print(f"    Region: x={bbox.get('x', 0)}, y={bbox.get('y', 0)}, "
          f"w={bbox.get('w', 0)}, h={bbox.get('h', 0)}")
    print()

# People detection
people = result.get("peopleResult", {}).get("values", [])
print(f"People detected: {len(people)}")
for person in people:
    bbox = person.get("boundingBox", {})
    print(f"  Confidence: {person.get('confidence', 0):.4f}")
    print(f"    Bounding box: x={bbox.get('x', 0)}, y={bbox.get('y', 0)}, "
          f"w={bbox.get('w', 0)}, h={bbox.get('h', 0)}")


## Summary

In this lab you:

- Constructed authenticated REST requests to the Image Analysis 4.0 endpoint
- Requested multiple visual features (tags, captions, objects, OCR, dense captions, people) in a single call
- Parsed the structured JSON response for each feature type
- Sent both URL-based and binary image inputs using the correct content types
